In [ ]:
import os
import time
import warnings
import argparse
from typing import Dict, Any, Tuple, List

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve, roc_curve, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

from paraboost_forest import ParaBoostForest


def choose_best_threshold_from_pr(y_true: np.ndarray, y_proba: np.ndarray) -> tuple[float, float]:
    """
    Returns (best_thr, best_f1) chosen directly from the Precision-Recall curve (max F1).
    Robust to extreme class imbalance and tiny probability scales.
    """
    from sklearn.metrics import precision_recall_curve
    prec, rec, thr_raw = precision_recall_curve(y_true, y_proba)
    # f1 برای اولین نقطه (rec=1, thr undefined) هم تعریف می‌شود؛ thr_raw یک عنصر کمتر از prec/rec دارد
    f1s = 2 * prec * rec / (prec + rec + 1e-12)
    if len(f1s) == 0:
        # fallback نادر
        return float(np.quantile(y_proba, 0.995)), 0.0
    best_idx = int(np.nanargmax(f1s))
    # نگاشت ایندکس به thr: برای idx=0، thr تعریف ندارد، پس به نزدیک‌ترین آستانه معتبر برگردیم
    if len(thr_raw) == 0:
        best_thr = float(np.quantile(y_proba, 0.995))
    else:
        j = min(len(thr_raw) - 1, max(0, best_idx - 1))
        best_thr = float(thr_raw[j])
    best_f1 = float(np.nanmax(f1s))
    return best_thr, best_f1


def cv_pr_auc(pipeline, X, y, cv) -> float:
    """Cross-val average precision without thresholding (pure ranking quality)."""
    scores = []
    for tr_idx, va_idx in cv.split(X, y):
        Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
        ytr, yva = y.iloc[tr_idx], y.iloc[va_idx]
        pipeline.fit(Xtr, ytr)
        clf = pipeline.named_steps['clf']
        if hasattr(clf, 'predict_proba'):
            y_proba = pipeline.predict_proba(Xva)[:, 1]
        elif hasattr(clf, 'decision_function'):
            s = pipeline.decision_function(Xva)
            y_proba = 1.0 / (1.0 + np.exp(-s))
        else:
            y_proba = pipeline.predict(Xva).astype(float)
        scores.append(average_precision_score(yva, y_proba))
    return float(np.mean(scores))

# Optional dependencies
try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner
except Exception as e:
    raise RuntimeError("Optuna مورد نیاز است. نصب: pip install optuna") from e

try:
    from xgboost import XGBClassifier  # type: ignore
except Exception:
    XGBClassifier = None
    warnings.warn("XGBoost نصب نیست؛ بخش XGBoost رد می‌شود. pip install xgboost")

try:
    from lightgbm import LGBMClassifier  # type: ignore
except Exception:
    LGBMClassifier = None
    warnings.warn("LightGBM نصب نیست؛ بخش LightGBM رد می‌شود. pip install lightgbm")

try:
    from imblearn.over_sampling import SMOTE, ADASYN  # type: ignore
    from imblearn.under_sampling import RandomUnderSampler  # type: ignore
    from imblearn.pipeline import Pipeline as ImbPipeline  # type: ignore
except Exception:
    SMOTE = ADASYN = RandomUnderSampler = None
    ImbPipeline = None
    warnings.warn("imbalanced-learn نصب نیست؛ SMOTE/ADASYN/Undersampling در دسترس نیست. pip install imbalanced-learn")

RANDOM_STATE = 42
OUTPUT_DIR = 'outputs'
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
RESULTS_CSV = os.path.join(OUTPUT_DIR, 'results.csv')


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(y_prob, bins) - 1
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        m = (idx == b)
        if not np.any(m):
            continue
        conf = float(y_prob[m].mean())
        acc = float(y_true[m].mean())
        ece += (m.mean()) * abs(acc - conf)
    return float(ece)


def precision_recall_at_k(y_true: np.ndarray, y_prob: np.ndarray, ks: Tuple[int, ...] = (50, 100, 200, 500)) -> Dict[str, float]:
    order = np.argsort(-y_prob)
    y_sorted = y_true[order]
    total_pos = float(y_true.sum()) + 1e-12
    out: Dict[str, float] = {}
    for k in ks:
        k_eff = min(int(k), len(y_sorted))
        tp = int(y_sorted[:k_eff].sum())
        out[f"P@{k}"] = tp / max(k_eff, 1)
        out[f"R@{k}"] = tp / total_pos
    return out


def lift_gains_points(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10):
    df = pd.DataFrame(dict(y=y_true, p=y_prob)).sort_values('p', ascending=False).reset_index(drop=True)
    df['bin'] = pd.qcut(df.index, n_bins, labels=False)
    tot_pos = float(df['y'].sum()) + 1e-12
    gains: List[float] = []
    lift: List[float] = []
    x: List[float] = []
    for b in range(n_bins):
        cum = float(df.loc[df['bin'] <= b, 'y'].sum())
        prop = (b + 1) / n_bins
        gains.append(cum / tot_pos)
        lift.append((cum / tot_pos) / prop)
        x.append(prop)
    return np.array(x), np.array(gains), np.array(lift)


def save_predictions(name: str, y_true, y_proba: np.ndarray, y_pred: np.ndarray, split: str = 'test') -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    arr_true = y_true.values if hasattr(y_true, 'values') else y_true
    df = pd.DataFrame({
        'id': np.arange(len(arr_true)),
        'y_true': arr_true,
        'y_pred_proba': y_proba,
        'y_pred_label': y_pred,
        'split': split,
    })
    df.to_csv(os.path.join(OUTPUT_DIR, f'predictions_{name}.csv'), index=False)


def bootstrap_ci(y_true: np.ndarray, y_prob: np.ndarray, metric_fn, n_boot: int = 1000, seed: int = 42) -> Tuple[float, float]:
    rng = np.random.RandomState(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        vals.append(float(metric_fn(y_true[idx], y_prob[idx])))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def set_random_state(seed: int) -> None:
    global RANDOM_STATE
    RANDOM_STATE = int(seed)


class CreditCardFeatureEngineer(BaseEstimator, TransformerMixin):
    """افزودن فیچرهای ساده برای Amount و Time (log1p)."""

    def __init__(self, amount_col: str = 'Amount', time_col: str = 'Time'):
        self.amount_col = amount_col
        self.time_col = time_col
        self.columns_: list[str] = []

    def fit(self, X: pd.DataFrame, y=None):
        self.columns_ = list(X.columns)
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.columns_ or None)
        df = X.copy()
        if self.amount_col in df.columns:
            df['Amount_log1p'] = np.log1p(df[self.amount_col].astype(float))
        if self.time_col in df.columns:
            df['Time_log1p'] = np.log1p(df[self.time_col].astype(float))
        return df.values.astype(float)


class PreprocessWrapper(BaseEstimator, TransformerMixin):
    """ترنسفورمر واحد که یک Pipeline داخلی (FE+Impute+Scale) را کپسوله می‌کند.
    این کار از خطای imblearn.Pipeline در مورد Nested Pipeline جلوگیری می‌کند.
    """
    def __init__(self, amount_col: str = 'Amount', time_col: str = 'Time'):
        self.amount_col = amount_col
        self.time_col = time_col
        self._pipe: Pipeline | None = None

    def fit(self, X: pd.DataFrame, y=None):
        fe = CreditCardFeatureEngineer(self.amount_col, self.time_col)
        self._pipe = Pipeline([
            ('fe', fe),
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ])
        self._pipe.fit(X, y)
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        if self._pipe is None:
            raise RuntimeError('PreprocessWrapper is not fitted')
        return self._pipe.transform(X)


def load_creditcard(csv_path: str) -> Tuple[pd.DataFrame, pd.Series]:
    path = csv_path if os.path.exists(csv_path) else os.path.basename(csv_path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"فایل پیدا نشد: {csv_path}")
    df = pd.read_csv(path)
    if 'Class' not in df.columns:
        raise ValueError("ستون هدف Class یافت نشد")
    y = df['Class'].astype(int)
    X = df.drop(columns=['Class'])
    # حذف ستون‌های دارای null (طبق دستور، فقط در صورت null)
    null_cols = [c for c in X.columns if X[c].isna().any()]
    if null_cols:
        warnings.warn(f"حذف ستون‌های دارای مقدار گمشده: {null_cols}")
        X = X.drop(columns=null_cols)
    return X, y


def build_preprocessor() -> PreprocessWrapper:
    return PreprocessWrapper(amount_col='Amount', time_col='Time')


def report_class_dist(y: pd.Series, title: str) -> None:
    counts = y.value_counts().to_dict()
    total = len(y)
    print(f"{title} class distribution: {counts}, pos_ratio={counts.get(1,0)/total:.6f}")


def stratified_subsample(X: pd.DataFrame, y: pd.Series, max_samples: int, seed: int) -> Tuple[pd.DataFrame, pd.Series]:
    if len(X) <= max_samples:
        return X, y
    X_sub, _, y_sub, _ = train_test_split(X, y, train_size=max_samples, stratify=y, random_state=seed)
    return X_sub, y_sub


def cv_score_with_thr(pipeline, X, y, cv) -> float:
    scores = []
    for tr_idx, va_idx in cv.split(X, y):
        Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
        ytr, yva = y.iloc[tr_idx], y.iloc[va_idx]
        pipeline.fit(Xtr, ytr)

        clf = pipeline.named_steps['clf']
        if hasattr(clf, 'predict_proba'):
            y_proba = pipeline.predict_proba(Xva)[:, 1]
        elif hasattr(clf, 'decision_function'):
            s = pipeline.decision_function(Xva)
            y_proba = 1.0 / (1.0 + np.exp(-s))
        else:
            # آخرین چاره؛ خروجی باینری را float کنیم
            y_proba = pipeline.predict(Xva).astype(float)

        # چاپ تشخیصی سبک
        print(f"[DEBUG] proba_stats: min={y_proba.min():.6g}, max={y_proba.max():.6g}, "
              f"mean={y_proba.mean():.6g}, pct>=0.1={(y_proba>=0.1).mean():.6g}")

        best_thr, best_f1 = choose_best_threshold_from_pr(yva.values, y_proba)
        scores.append(best_f1)
    return float(np.mean(scores))


def build_pipeline(pre, estimator, sampler_name: str) -> Pipeline:
    steps = [('pre', pre)]
    if ImbPipeline is None:
        steps.append(('clf', estimator))
        return Pipeline(steps)
    sampler = None
    if sampler_name == 'smote' and SMOTE is not None:
        sampler = SMOTE(random_state=RANDOM_STATE)
    elif sampler_name == 'adasyn' and ADASYN is not None:
        sampler = ADASYN(random_state=RANDOM_STATE)
    elif sampler_name == 'under' and RandomUnderSampler is not None:
        sampler = RandomUnderSampler(random_state=RANDOM_STATE)
    if sampler is not None:
        steps.append(('sampler', sampler))
    steps.append(('clf', estimator))
    return ImbPipeline(steps) if ImbPipeline is not None else Pipeline(steps)


def optimize_pbf(X, y, pre, trials: int, tuning_max_samples: int = 80000):
    X_tune, y_tune = stratified_subsample(X, y, tuning_max_samples, RANDOM_STATE)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial: 'optuna.Trial') -> float:
        n_estimators = trial.suggest_categorical('n_estimators', [100, 200, 300])
        n_tps = trial.suggest_categorical('n_trees_per_stage', [3, 5, 7])
        lr = trial.suggest_float('learning_rate', 5e-3, 0.2, log=True)
        max_depth = trial.suggest_int('max_depth', 4, 10)
        max_features = trial.suggest_categorical('max_features', ['sqrt', 0.5, 0.7])
        class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])
        sampler_name = trial.suggest_categorical('sampler', ['none', 'smote', 'under'])

        clf = ParaBoostForest(
            task='classification', n_estimators=n_estimators, n_trees_per_stage=n_tps,
            learning_rate=lr, max_depth=max_depth, max_features=max_features,
            class_weight=class_weight, early_stopping_rounds=10, validation_fraction=0.1,
            early_stop_metric='auc', n_jobs=-1, random_state=RANDOM_STATE, verbose=0,
            enable_resampling=False,
        )
        pipe = build_pipeline(pre, clf, sampler_name)
        return cv_pr_auc(pipe, X_tune, y_tune, cv)

    sampler = TPESampler(seed=RANDOM_STATE)
    pruner = MedianPruner(n_startup_trials=5)
    study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=trials, show_progress_bar=False, timeout=600)
    return study


def optimize_xgb(X, y, pre, trials: int, tuning_max_samples: int = 40000):
    if XGBClassifier is None:
        return None
    X_tune, y_tune = stratified_subsample(X, y, tuning_max_samples, RANDOM_STATE)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    pos_ratio = (y_tune == 1).mean()
    neg_pos_ratio = float((1.0 - pos_ratio) / max(pos_ratio, 1e-9))

    def objective(trial: 'optuna.Trial') -> float:
        n_estimators = trial.suggest_categorical('n_estimators', [200, 400])
        max_depth = trial.suggest_int('max_depth', 3, 8)
        learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.2, log=True)
        min_child_weight = trial.suggest_int('min_child_weight', 1, 8)
        subsample = trial.suggest_float('subsample', 0.7, 1.0)
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.7, 1.0)
        reg_lambda = trial.suggest_float('reg_lambda', 1e-3, 5.0, log=True)
        scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, max(1.0, neg_pos_ratio), log=True)
        sampler_name = trial.suggest_categorical('sampler', ['none', 'smote', 'under'])

        clf = XGBClassifier(
            n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
            min_child_weight=min_child_weight, subsample=subsample, colsample_bytree=colsample_bytree,
            reg_lambda=reg_lambda, scale_pos_weight=scale_pos_weight, n_jobs=-1, tree_method='hist',
            eval_metric='logloss', random_state=RANDOM_STATE,
        )
        pipe = build_pipeline(pre, clf, sampler_name)
        return cv_pr_auc(pipe, X_tune, y_tune, cv)

    sampler = TPESampler(seed=RANDOM_STATE)
    pruner = MedianPruner(n_startup_trials=5)
    study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=trials, show_progress_bar=False, timeout=600)
    return study


def optimize_rf(X, y, pre, trials: int, tuning_max_samples: int = 40000):
    X_tune, y_tune = stratified_subsample(X, y, tuning_max_samples, RANDOM_STATE)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial: 'optuna.Trial') -> float:
        n_estimators = trial.suggest_categorical('n_estimators', [200, 500])
        max_depth = trial.suggest_categorical('max_depth', [None, 8, 12])
        max_features = trial.suggest_categorical('max_features', ['sqrt', 0.5])
        min_samples_leaf = trial.suggest_categorical('min_samples_leaf', [1, 2])
        class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])
        sampler_name = trial.suggest_categorical('sampler', ['none', 'smote', 'under'])

        clf = RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth, max_features=max_features,
            min_samples_leaf=min_samples_leaf, class_weight=class_weight,
            n_jobs=-1, random_state=RANDOM_STATE,
        )
        pipe = build_pipeline(pre, clf, sampler_name)
        return cv_pr_auc(pipe, X_tune, y_tune, cv)

    sampler = TPESampler(seed=RANDOM_STATE)
    pruner = MedianPruner(n_startup_trials=5)
    study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=trials, show_progress_bar=False, timeout=600)
    return study


def fit_with_threshold(pre, study, model_key: str, X_train, y_train, X_valid, y_valid):
    params = dict(study.best_params)
    sampler_name = params.pop('sampler', 'none')
    if model_key == 'pbf':
        clf = ParaBoostForest(task='classification', n_jobs=-1, random_state=RANDOM_STATE,
                              early_stopping_rounds=10, validation_fraction=0.1,
                              early_stop_metric='auc', verbose=0, enable_resampling=False,
                              **params)
    elif model_key == 'xgb':
        if XGBClassifier is None:
            return None, 0.5, float('nan')
        clf = XGBClassifier(n_jobs=-1, tree_method='hist', eval_metric='logloss', random_state=RANDOM_STATE,
                            **params)
    else:
        clf = RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE, **params)

    pipe = build_pipeline(pre, clf, sampler_name)
    t0 = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - t0

    thr = 0.5
    y_proba = None
    clf_est = pipe.named_steps['clf']
    if hasattr(clf_est, 'predict_proba'):
        y_proba = pipe.predict_proba(X_valid)[:, 1]
    elif hasattr(clf_est, 'decision_function'):
        s = pipe.decision_function(X_valid)
        y_proba = 1.0 / (1.0 + np.exp(-s))
    else:
        y_proba = pipe.predict(X_valid).astype(float)

    # --- Calibration (only for PBF, RF) ---
    if model_key in ('pbf', 'rf'):
        try:
            from sklearn.isotonic import IsotonicRegression
            calib = IsotonicRegression(out_of_bounds='clip')
            calib.fit(y_proba, y_valid.values)
            setattr(pipe, '_calibrator', calib)
            y_proba = calib.predict(y_proba)
        except Exception:
            pass

    # چاپ تشخیصی سبک
    print(f"[DEBUG] proba_stats: min={y_proba.min():.6g}, max={y_proba.max():.6g}, "
          f"mean={y_proba.mean():.6g}, pct>=0.1={(y_proba>=0.1).mean():.6g}")

    # --- انتخاب آستانه از PR-curve ---
    thr, _ = choose_best_threshold_from_pr(y_valid.values, y_proba)

    # sanity-check: حداقل نرخ مثبت را تضمین کن
    min_rate = max(5 / len(y_valid), 1e-6)
    if (y_proba >= thr).mean() < min_rate:
        thr = float(np.quantile(y_proba, 1 - min_rate))
    print(f"[THR] {model_key}: chosen={thr:.6g}, pos_rate@thr={(y_proba>=thr).mean():.6g}, max_proba={y_proba.max():.6g}")
    return pipe, thr, train_time


def evaluate(pipe, thr, X_test, y_test, name: str) -> Dict[str, Any]:
    t1 = time.time()
    clf = pipe.named_steps['clf']
    y_proba = None
    if hasattr(clf, 'predict_proba'):
        y_proba = pipe.predict_proba(X_test)[:, 1]
    elif hasattr(clf, 'decision_function'):
        s = pipe.decision_function(X_test)
        y_proba = 1.0 / (1.0 + np.exp(-s))
    else:
        y_proba = pipe.predict(X_test).astype(float)
    # چاپ تشخیصی سبک
    print(f"[DEBUG] proba_stats: min={y_proba.min():.6g}, max={y_proba.max():.6g}, "
          f"mean={y_proba.mean():.6g}, pct>=0.1={(y_proba>=0.1).mean():.6g}")
    # اگر آستانه خیلی سخت شد و هیچ مثبت نداشتیم، با ملایمت تعدیل کن
    if (y_proba >= thr).mean() < 1e-9:
        thr = float(np.quantile(y_proba, 0.999))
    y_pred = (y_proba >= thr).astype(int)
    pred_time = time.time() - t1

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else float('nan'),
        'pr_auc': average_precision_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else float('nan'),
        'threshold': thr,
        'pred_time_s': pred_time,
    }
    try:
        from sklearn.metrics import brier_score_loss
        from scipy.stats import ks_2samp
        metrics['brier'] = brier_score_loss(y_test, y_proba)
        ks = ks_2samp(y_proba[y_test==1], y_proba[y_test==0])
        metrics['ks_stat'] = float(ks.statistic)
        metrics['ks_pvalue'] = float(ks.pvalue)
        # ECE and fixed-threshold metrics
        arr_true = y_test.values if hasattr(y_test, 'values') else y_test
        metrics['ece'] = expected_calibration_error(arr_true, y_proba)
        y_pred_fixed = (y_proba >= 0.5).astype(int)
        metrics['f1_at_fixed'] = f1_score(y_test, y_pred_fixed, zero_division=0)
        metrics['precision_at_fixed'] = precision_score(y_test, y_pred_fixed, zero_division=0)
        metrics['recall_at_fixed'] = recall_score(y_test, y_pred_fixed, zero_division=0)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_fixed).ravel()
        metrics['fn_at_fixed'] = int(fn); metrics['fp_at_fixed'] = int(fp)
        tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_pred).ravel()
        metrics['fpr_at_thr'] = fp_t / max(fp_t + tn_t, 1)
        metrics['fnr_at_thr'] = fn_t / max(fn_t + tp_t, 1)
        metrics.update(precision_recall_at_k(arr_true, y_proba))
    except Exception:
        pass
    try:
        import matplotlib.pyplot as plt
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        prec, rec, _ = precision_recall_curve(y_test, y_proba)
        os.makedirs(PLOTS_DIR, exist_ok=True)
        plt.figure(figsize=(5,4))
        plt.plot(fpr, tpr, label=f"AUC={metrics['roc_auc']:.4f}")
        plt.plot([0,1],[0,1],'k--',alpha=0.4)
        plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title(f'ROC - {name}'); plt.legend(); plt.tight_layout()
        roc_path = os.path.join(PLOTS_DIR, f"roc_{name.replace(' ','_')}.png"); plt.savefig(roc_path); plt.close()
        plt.figure(figsize=(5,4))
        plt.plot(rec, prec, label=f"PR-AUC={metrics['pr_auc']:.4f}")
        plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title(f'PR - {name}'); plt.legend(); plt.tight_layout()
        pr_path = os.path.join(PLOTS_DIR, f"pr_{name.replace(' ','_')}.png"); plt.savefig(pr_path); plt.close()
        metrics['roc_plot'] = roc_path
        metrics['pr_plot'] = pr_path
    except Exception as e:
        warnings.warn(f"عدم امکان ترسیم نمودار: {e}")
    return metrics


def comparative_plots(models: Dict[str, Tuple[Pipeline, float]], X_test: pd.DataFrame, y_test: pd.Series) -> Dict[str, str]:
    """رسم نمودارهای مقایسه‌ای برای مقاله: ROC/PR مشترک، F1-Threshold، Calibration، Confusion Matrix و Bar chart.
    خروجی: مسیر فایل تصاویر ذخیره شده.
    """
    paths: Dict[str, str] = {}
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns  # type: ignore
        from sklearn.metrics import confusion_matrix

        # جمع‌آوری پیش‌بینی احتمالات
        proba_dict: Dict[str, np.ndarray] = {}
        for name, (pipe, thr) in models.items():
            clf = pipe.named_steps['clf']
            if hasattr(clf, 'predict_proba'):
                proba = pipe.predict_proba(X_test)[:, 1]
            elif hasattr(clf, 'decision_function'):
                s = pipe.decision_function(X_test)
                proba = 1.0 / (1.0 + np.exp(-s))
            else:
                proba = pipe.predict(X_test).astype(float)
            proba_dict[name] = proba

        # 1) ROC مقایسه‌ای
        plt.figure(figsize=(6, 5))
        for name, proba in proba_dict.items():
            fpr, tpr, _ = roc_curve(y_test, proba)
            aucv = roc_auc_score(y_test, proba) if len(np.unique(y_test)) > 1 else float('nan')
            plt.plot(fpr, tpr, label=f"{name} (AUC={aucv:.4f})")
        plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
        plt.xlabel('FPR')
        plt.ylabel('TPR')
        plt.title('ROC Comparison')
        plt.legend()
        plt.tight_layout()
        os.makedirs(PLOTS_DIR, exist_ok=True)
        p = os.path.join(PLOTS_DIR, 'cmp_roc.png'); plt.savefig(p); plt.close(); paths['roc'] = p

        # 2) PR مقایسه‌ای
        plt.figure(figsize=(6, 5))
        for name, proba in proba_dict.items():
            prec, rec, _ = precision_recall_curve(y_test, proba)
            pr_auc = average_precision_score(y_test, proba) if len(np.unique(y_test)) > 1 else float('nan')
            plt.plot(rec, prec, label=f"{name} (AP={pr_auc:.4f})")
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('PR Comparison')
        plt.legend()
        plt.tight_layout()
        p = os.path.join(PLOTS_DIR, 'cmp_pr.png'); plt.savefig(p); plt.close(); paths['pr'] = p

        # 3) F1 در برابر Threshold
        plt.figure(figsize=(6, 5))
        for name, proba in proba_dict.items():
            thr_grid = np.unique(np.r_[1e-6, np.quantile(proba, np.linspace(0.001, 0.999, 60))])
            f1s = []
            for t in thr_grid:
                y_pred = (proba >= t).astype(int)
                f1s.append(f1_score(y_test, y_pred, zero_division=0))
            plt.plot(thr_grid, f1s, label=name)
        plt.xlabel('Threshold')
        plt.ylabel('F1')
        plt.title('F1 vs Threshold')
        plt.legend()
        plt.tight_layout()
        p = os.path.join(PLOTS_DIR, 'cmp_f1_threshold.png'); plt.savefig(p); plt.close(); paths['f1_thr'] = p

        # 4) Calibration curves
        plt.figure(figsize=(6, 5))
        for name, proba in proba_dict.items():
            prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=15, strategy='quantile')
            plt.plot(prob_pred, prob_true, marker='o', label=name)
        plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
        plt.xlabel('Mean Predicted Probability')
        plt.ylabel('Fraction of Positives')
        plt.title('Calibration Curves')
        plt.legend()
        plt.tight_layout()
        p = os.path.join(PLOTS_DIR, 'cmp_calibration.png'); plt.savefig(p); plt.close(); paths['cal'] = p

        # Helper: Precision@K / Recall@K
        def precision_recall_at_k(y_true, y_proba, ks=(20, 50, 100, 200)):
            order = np.argsort(-y_proba)
            y_sorted = y_true.iloc[order].values if hasattr(y_true, 'iloc') else y_true[order]
            res = []
            for k in ks:
                k = min(k, len(y_sorted))
                tp = int(y_sorted[:k].sum())
                prec_k = tp / max(k, 1)
                rec_k = tp / max(int(y_true.sum()), 1)
                res.append((k, prec_k, rec_k))
            return res

        # 5) Confusion matrices at chosen thresholds
        fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
        if len(models) == 1:
            axes = [axes]
        for ax, (name, (pipe, thr)) in zip(axes, models.items()):
            proba = proba_dict[name]
            y_pred = (proba >= thr).astype(int)
            cm = confusion_matrix(y_test, y_pred)
            sns.heatmap(cm, annot=True, fmt='d', cbar=False, ax=ax)
            ax.set_title(f'{name} @thr={thr:.2f}')
            ax.set_xlabel('Predicted')
            ax.set_ylabel('True')
        plt.tight_layout()
        p = os.path.join(PLOTS_DIR, 'cmp_confusion.png'); fig.savefig(p); plt.close(fig); paths['cm'] = p

        # 6) Precision@K plot (log-K)
        plt.figure(figsize=(7, 5))
        for name, proba in proba_dict.items():
            rows = precision_recall_at_k(y_test, proba)
            ks = [r[0] for r in rows]
            preks = [r[1] for r in rows]
            plt.plot(ks, preks, marker='o', label=f'{name} P@K')
        plt.xscale('log'); plt.xlabel('K'); plt.ylabel('Precision@K'); plt.title('Precision@K (log-K)')
        plt.legend(); plt.tight_layout()
        p = os.path.join(PLOTS_DIR, 'cmp_p_at_k.png'); plt.savefig(p); plt.close(); paths['p_at_k'] = p

        # 7) Recall@K plot (log-K)
        plt.figure(figsize=(7, 5))
        for name, proba in proba_dict.items():
            rows = precision_recall_at_k(y_test, proba)
            ks = [r[0] for r in rows]
            rcks = [r[2] for r in rows]
            plt.plot(ks, rcks, marker='o', label=f'{name} R@K')
        plt.xscale('log'); plt.xlabel('K'); plt.ylabel('Recall@K'); plt.title('Recall@K (log-K)')
        plt.legend(); plt.tight_layout()
        p = os.path.join(PLOTS_DIR, 'cmp_r_at_k.png'); plt.savefig(p); plt.close(); paths['r_at_k'] = p

        # 8) Probability histogram (log-density)
        plt.figure(figsize=(7, 5))
        for name, proba in proba_dict.items():
            plt.hist(proba, bins=50, histtype='step', density=True, label=name)
        plt.yscale('log'); plt.xlabel('Predicted probability'); plt.ylabel('Density (log)')
        plt.title('Probability Distribution (log-density)'); plt.legend(); plt.tight_layout()
        p = os.path.join(PLOTS_DIR, 'cmp_proba_hist.png'); plt.savefig(p); plt.close(); paths['proba_hist'] = p

        # 9) Combined P@K and R@K panel
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        for name, proba in proba_dict.items():
            rows = precision_recall_at_k(y_test, proba)
            ks = [r[0] for r in rows]
            preks = [r[1] for r in rows]
            rcks = [r[2] for r in rows]
            axes[0].plot(ks, preks, marker='o', label=name)
            axes[1].plot(ks, rcks, marker='o', label=name)
        for ax, title in zip(axes, ['Precision@K', 'Recall@K']):
            ax.set_xscale('log'); ax.set_xlabel('K'); ax.set_title(title); ax.legend()
        axes[0].set_ylabel('Precision'); axes[1].set_ylabel('Recall')
        fig.tight_layout()
        p = os.path.join(PLOTS_DIR, 'cmp_pr_at_k_panel.png'); fig.savefig(p); plt.close(fig); paths['pr_at_k_panel'] = p

        # 10) Cumulative gains and lift curves
        plt.figure(figsize=(6, 5))
        for name, proba in proba_dict.items():
            x, gains, lift = lift_gains_points(y_test.values if hasattr(y_test,'values') else y_test, proba, n_bins=10)
            plt.plot(x, gains, marker='o', label=name)
        plt.plot([0, 1], [0, 1], '--', alpha=0.4)
        plt.xlabel('Proportion of data (sorted by score)'); plt.ylabel('Cumulative proportion of positives')
        plt.title('Cumulative Gains'); plt.legend(); plt.tight_layout()
        p = os.path.join(PLOTS_DIR, 'cumulative_gains.png'); plt.savefig(p); plt.close(); paths['gains'] = p

        plt.figure(figsize=(6, 5))
        for name, proba in proba_dict.items():
            x, gains, lift = lift_gains_points(y_test.values if hasattr(y_test,'values') else y_test, proba, n_bins=10)
            plt.plot(x, lift, marker='o', label=name)
        plt.axhline(1.0, linestyle='--', alpha=0.4)
        plt.xlabel('Proportion of data (sorted by score)'); plt.ylabel('Lift')
        plt.title('Lift Curves'); plt.legend(); plt.tight_layout()
        p = os.path.join(PLOTS_DIR, 'lift_curves.png'); plt.savefig(p); plt.close(); paths['lift'] = p

    except Exception as e:
        warnings.warn(f'خطا در رسم نمودارهای مقایسه‌ای: {e}')
    return paths


def main():
    # تنظیم مسیر داده
    csv_path = 'creditcard.csv'  # در صورت قرارگیری در مسیر دیگر، این مقدار را تغییر دهید
    X, y = load_creditcard(csv_path)

    print('Shape:', X.shape)
    report_class_dist(y, 'Overall')

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    report_class_dist(y_train, 'Train')
    report_class_dist(y_test, 'Test')

    pre = build_preprocessor()

    # Trials برابر برای عدالت (قابل override با ENV: OPTUNA_TRIALS)
    trials = int(os.getenv('OPTUNA_TRIALS', '20'))

    print('Optuna: ParaBoostForest ...')
    study_pbf = optimize_pbf(X_train, y_train, pre, trials)
    print('Best PBF:', study_pbf.best_params, 'score=', study_pbf.best_value)

    if XGBClassifier is not None:
        print('Optuna: XGBoost ...')
        study_xgb = optimize_xgb(X_train, y_train, pre, trials)
        print('Best XGB:', study_xgb.best_params, 'score=', study_xgb.best_value)
    else:
        study_xgb = None
        print('XGBoost در دسترس نیست')

    print('Optuna: RandomForest ...')
    study_rf = optimize_rf(X_train, y_train, pre, trials)
    print('Best RF:', study_rf.best_params, 'score=', study_rf.best_value)

    # Threshold tuning روی یک ولیدیشن کوچک از Train
    X_tr_in, X_va_in, y_tr_in, y_va_in = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
    )

    print('Final fit & threshold tuning ...')
    pbf_pipe, pbf_thr, pbf_train_time = fit_with_threshold(pre, study_pbf, 'pbf', X_tr_in, y_tr_in, X_va_in, y_va_in)
    xgb_pipe = None; xgb_thr = 0.5; xgb_train_time = float('nan')
    if study_xgb is not None:
        xgb_pipe, xgb_thr, xgb_train_time = fit_with_threshold(pre, study_xgb, 'xgb', X_tr_in, y_tr_in, X_va_in, y_va_in)
    rf_pipe, rf_thr, rf_train_time = fit_with_threshold(pre, study_rf, 'rf', X_tr_in, y_tr_in, X_va_in, y_va_in)

    print('Evaluate on Test ...')
    res_pbf = evaluate(pbf_pipe, pbf_thr, X_test, y_test, 'ParaBoostForest'); res_pbf['train_time_s'] = pbf_train_time
    res_rf = evaluate(rf_pipe, rf_thr, X_test, y_test, 'RandomForest'); res_rf['train_time_s'] = rf_train_time
    if xgb_pipe is not None:
        res_xgb = evaluate(xgb_pipe, xgb_thr, X_test, y_test, 'XGBoost'); res_xgb['train_time_s'] = xgb_train_time
    else:
        res_xgb = None

    def fmt(r: Dict[str, Any]):
        return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items() if k not in {'roc_plot','pr_plot'}}

    print('\nResults (Test):')
    print('ParaBoostForest:', fmt(res_pbf))
    print('RandomForest:', fmt(res_rf))
    if res_xgb is not None:
        print('XGBoost:', fmt(res_xgb))

    # جمع‌بندی بر اساس F1 و ROC_AUC
    comp = [
        ('ParaBoostForest', res_pbf['f1'], res_pbf['roc_auc']),
        ('RandomForest', res_rf['f1'], res_rf['roc_auc'])
    ]
    if res_xgb is not None:
        comp.append(('XGBoost', res_xgb['f1'], res_xgb['roc_auc']))
    comp_sorted = sorted(comp, key=lambda x: (x[1], x[2]), reverse=True)
    print('\nRanking by (F1, ROC_AUC):')
    for name, f1v, aucv in comp_sorted:
        print(f"{name}: F1={f1v:.4f}, ROC_AUC={aucv:.4f}")

    # نمودارهای مقایسه‌ای برای مقاله
    models: Dict[str, Tuple[Pipeline, float]] = {
        'ParaBoostForest': (pbf_pipe, pbf_thr),
        'RandomForest': (rf_pipe, rf_thr),
    }
    if xgb_pipe is not None:
        models['XGBoost'] = (xgb_pipe, xgb_thr)
    cmp_paths = comparative_plots(models, X_test, y_test)
    print('Saved comparative plots:', cmp_paths)

    # Save results CSV
    import csv
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    rows = []
    rows.append({'model': 'ParaBoostForest', **{k: v for k, v in res_pbf.items() if k not in ('roc_plot', 'pr_plot')}})
    rows.append({'model': 'RandomForest', **{k: v for k, v in res_rf.items() if k not in ('roc_plot', 'pr_plot')}})
    if res_xgb is not None:
        rows.append({'model': 'XGBoost', **{k: v for k, v in res_xgb.items() if k not in ('roc_plot', 'pr_plot')}})
    fieldnames = sorted({k for r in rows for k in r.keys()})
    with open(RESULTS_CSV, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)
    print('Saved results to', RESULTS_CSV)


if __name__ == '__main__':
    main()